In [1]:
import pandas as pd 
import numpy as np 

In [2]:
df=pd.read_csv('healthcare_fraud_detection.csv')

In [3]:
df.head(10)

,Provider_ID,Claim_ID,Patient_Age,Patient_Gender,Diagnosis_Code,Procedure_Code,Claim_Amount,Approved_Amount,Insurance_Type,Claim_Submission_Date,Days_Between_Service_and_Claim,Number_of_Claims_Per_Provider_Monthly,Provider_Specialty,Patient_State,Claim_Status,Is_Fraud,Length_of_Stay,Visit_Type,Chronic_Condition_Flag,Prior_Visits_12m
0,P0052,C0000000,37,Male,I25.10,36415,443.51,393.16,Medicaid,2024-09-01,13,70,Cardiology,NY,Approved,0,0,Outpatient,1,2.0
1,P0121,C0000001,21,Female,E11.9,99213,467.50,461.33,Self-Pay,2022-09-05,5,62,General Practice,IL,Pending,0,5,Inpatient,1,2.0
2,P0140,C0000002,78,Female,J06.9,93000,591.69,530.06,Medicaid,2022-04-11,29,60,Cardiology,IL,Pending,0,5,Inpatient,1,3.0
3,P0202,C0000003,65,Male,I10,93000,235.15,189.11,Private,2023-10-11,22,70,General Practice,TX,Approved,0,0,Emergency,0,5.0
4,P0135,C0000004,36,Male,M54.5,85025,487.96,369.91,Private,2023-09-05,21,67,Pulmonology,PA,Approved,0,5,Inpatient,0,4.0
5,P0241,C0000005,44,Female,E11.9,80053,220.99,174.05,Medicaid,2024-12-20,26,74,Orthopedics,CA,Approved,0,2,Emergency,0,1.0
6,P0196,C0000006,50,Male,E78.5,93000,458.60,372.31,Medicare,2022-02-20,2,84,Cardiology,OH,Approved,0,0,Outpatient,1,0.0
7,P0071,C0000007,1,Male,I10,85025,934.34,321.60,NaN,2022-07-13,6,114,NaN,PA,Pending,1,2,Emergency,0,NaN
8,P0097,C0000008,34,Female,M54.5,97110,478.11,412.48,Medicare,2021-05-25,24,65,Pulmonology,IL,Pending,0,5,Inpatient,1,4.0
9,P0125,C0000009,47,Male,K21.9,99213,426.16,335.80,Self-Pay,2022-05-29,5,56,Internal Medicine,TX,Approved,0,2,Emergency,0,4.0


In [4]:
df.columns.tolist()  

['Provider_ID',
 'Claim_ID',
 'Patient_Age',
 'Patient_Gender',
 'Diagnosis_Code',
 'Procedure_Code',
 'Claim_Amount',
 'Approved_Amount',
 'Insurance_Type',
 'Claim_Submission_Date',
 'Days_Between_Service_and_Claim',
 'Number_of_Claims_Per_Provider_Monthly',
 'Provider_Specialty',
 'Patient_State',
 'Claim_Status',
 'Is_Fraud',
 'Length_of_Stay',
 'Visit_Type',
 'Chronic_Condition_Flag',
 'Prior_Visits_12m']

In [5]:
df.isnull().sum()

Provider_ID                                0
Claim_ID                                   0
Patient_Age                                0
Patient_Gender                             0
Diagnosis_Code                             0
Procedure_Code                             0
Claim_Amount                               0
Approved_Amount                            0
Insurance_Type                           350
Claim_Submission_Date                      0
Days_Between_Service_and_Claim             0
Number_of_Claims_Per_Provider_Monthly      0
Provider_Specialty                       350
Patient_State                              0
Claim_Status                               0
Is_Fraud                                   0
Length_of_Stay                             0
Visit_Type                                 0
Chronic_Condition_Flag                     0
Prior_Visits_12m                         350
dtype: int64

In [6]:
#  here we check if missing data was happen due to transfer or related to each other such as same row
mask = df['Insurance_Type'].isnull()
df[mask][['Provider_Specialty','Prior_Visits_12m']].isnull().sum() 

Provider_Specialty    15
Prior_Visits_12m      12
dtype: int64

In [7]:
df.groupby(df['Insurance_Type'].isnull())['Is_Fraud'].mean()

Insurance_Type
False    0.083316
True     0.071429
Name: Is_Fraud, dtype: float64

In [8]:
df['Insurance_Type']=df['Insurance_Type'].fillna('Unknown')

In [9]:
df.groupby(df['Prior_Visits_12m'].isnull())['Is_Fraud'].mean()

Prior_Visits_12m
False    0.082902
True     0.082857
Name: Is_Fraud, dtype: float64

In [10]:
df['Prior_Visits_12m'] = df['Prior_Visits_12m'].fillna(df['Prior_Visits_12m'].median())

In [11]:
# 1. Build a lookup: each provider → their known specialty
lookup = (df.dropna(subset=['Provider_Specialty'])
            .groupby('Provider_ID')['Provider_Specialty']
            .first())

# 2. Fill blanks from the lookup, then "Unknown" for the rest
df['Provider_Specialty'] = df['Provider_Specialty'].fillna(df['Provider_ID'].map(lookup))
df['Provider_Specialty'] = df['Provider_Specialty'].fillna('Unknown')

In [12]:
df.isnull().sum()

Provider_ID                              0
Claim_ID                                 0
Patient_Age                              0
Patient_Gender                           0
Diagnosis_Code                           0
Procedure_Code                           0
Claim_Amount                             0
Approved_Amount                          0
Insurance_Type                           0
Claim_Submission_Date                    0
Days_Between_Service_and_Claim           0
Number_of_Claims_Per_Provider_Monthly    0
Provider_Specialty                       0
Patient_State                            0
Claim_Status                             0
Is_Fraud                                 0
Length_of_Stay                           0
Visit_Type                               0
Chronic_Condition_Flag                   0
Prior_Visits_12m                         0
dtype: int64

In [13]:
#Question 1: "Which providers are fraud outliers?"

provider_stats = df.groupby('Provider_ID').agg(
    total_claims=('Claim_ID', 'count'),      # how many receipts are in this folder?
    fraud_claims=('Is_Fraud', 'sum'),        # Is_Fraud is 0/1, so sum = count of fraud claims
    fraud_rate=('Is_Fraud', 'mean')          # mean of 0/1 column = % that are fraud
).reset_index()

In [14]:
provider_stats.sort_values('fraud_rate', ascending=False).head(10)

,Provider_ID,total_claims,fraud_claims,fraud_rate
86,P0086,38,16,0.421053
159,P0159,20,7,0.350000
110,P0110,45,15,0.333333
104,P0104,34,11,0.323529
223,P0223,44,14,0.318182
293,P0293,33,10,0.303030
255,P0255,28,8,0.285714
27,P0027,37,10,0.270270
71,P0071,32,8,0.250000
101,P0101,36,9,0.250000


In [15]:
## the trap: filter for reliability.A provider with 2 claims where 1 is fraud has a "50% fraud rate" — but that's just 1 unlucky claim, not a pattern. Before trusting a rate, filter for a minimum sample size:
provider_stats[provider_stats['total_claims']>=10].sort_values('fraud_rate', ascending=False).head(10)

,Provider_ID,total_claims,fraud_claims,fraud_rate
86,P0086,38,16,0.421053
159,P0159,20,7,0.350000
110,P0110,45,15,0.333333
104,P0104,34,11,0.323529
223,P0223,44,14,0.318182
293,P0293,33,10,0.303030
255,P0255,28,8,0.285714
27,P0027,37,10,0.270270
71,P0071,32,8,0.250000
101,P0101,36,9,0.250000


In [16]:
##Question 2: "Should this claim auto-approve or get held for review?" (Claims Ops)

df['days_bucket'] = pd.cut(
    df['Days_Between_Service_and_Claim'],
    bins=[-1, 7, 14, 21, 30, 100],           # the cut points
    labels=['0-7','8-14','15-21','22-30','31+']  # human-readable names for each bucket
)

In [17]:
df.groupby('days_bucket', observed=True)['Is_Fraud'].agg(['count','mean'])

,count,mean
days_bucket,,
0-7,2827,0.293244
8-14,2287,0.000000
15-21,2264,0.000000
22-30,2622,0.000000


In [18]:
specialty_stats = df.groupby('Provider_Specialty').agg(
    total_claims=('Claim_ID', 'count'),
    fraud_claims=('Is_Fraud', 'sum'),
    fraud_rate=('Is_Fraud', 'mean')
).round(3).sort_values('fraud_rate', ascending=False)

specialty_stats['diff_from_baseline_pts'] = ((specialty_stats['fraud_rate'] - df['Is_Fraud'].mean()) * 100).round(1)

specialty_stats

,total_claims,fraud_claims,fraud_rate,diff_from_baseline_pts
Provider_Specialty,,,,
General Practice,1943,188,0.097,1.4
Internal Medicine,2087,180,0.086,0.3
Orthopedics,1677,144,0.086,0.3
Pulmonology,1303,99,0.076,-0.7
Cardiology,1406,105,0.075,-0.8
Neurology,1584,113,0.071,-1.2


In [19]:
#Question 3: "How much dollar exposure are we carrying from fraud?" (Finance/Compliance)
total_dollars = df['Claim_Amount'].sum()                          # add up every claim's dollar amount
fraud_dollars = df[df['Is_Fraud']==1]['Claim_Amount'].sum()        # filter to fraud claims only, THEN sum
fraud_dollar_share = fraud_dollars / total_dollars                 # what fraction of total $ is fraud

fraud_dollar_share 


0.14341413079144505

In [23]:
provider_stats = df.groupby('Provider_ID').agg(
    total_claims=('Claim_ID', 'count'),
    fraud_rate=('Is_Fraud', 'mean')
).reset_index()

top10_ids = provider_stats.sort_values('fraud_rate', ascending=False).head(10)['Provider_ID']

provider_stats

,Provider_ID,total_claims,fraud_rate
0,P0000,40,0.075000
1,P0001,36,0.111111
2,P0002,34,0.235294
3,P0003,33,0.090909
4,P0004,25,0.040000
...,...,...,...
295,P0295,36,0.027778
296,P0296,36,0.027778
297,P0297,37,0.135135
298,P0298,27,0.037037


In [24]:
top10_claims = df[df['Provider_ID'].isin(top10_ids)]
exposure = top10_claims['Claim_Amount'].sum()